# Generate Task 2 embeddings — Google Colab (no token needed)

Do **Runtime -> Run all**. No GitHub token, no cloning. It downloads the data and
the two models, builds the embeddings, and downloads a file **task2_embeddings.zip**
to your computer. Then upload that zip to Claude and it gets pushed to your repo.

Each step prints clear progress.


## STEP 1 of 3 — Install libraries


In [ ]:
print("STEP 1 of 3 - installing libraries (about 1 minute) ...")
import sys, subprocess
rc = subprocess.run([sys.executable,"-m","pip","install","-q",
                     "sentence-transformers","datasets","pandas","numpy"]).returncode
print("\u2705 STEP 1 DONE - libraries ready." if rc==0 else "\u274c install failed")


## STEP 2 of 3 — Download data + models and build embeddings


In [ ]:
print("STEP 2 of 3 - building embeddings\n")
import os, json, time
import numpy as np, pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

os.makedirs("out", exist_ok=True)
print("\u23f3 downloading SciFact data ...")
corpus  = load_dataset("BeIR/scifact","corpus",split="corpus").to_pandas().rename(columns={"_id":"doc_id"})
queries = load_dataset("BeIR/scifact","queries",split="queries").to_pandas().rename(columns={"_id":"query_id"})
qrels   = load_dataset("BeIR/scifact-qrels",split="test").to_pandas()
qrels.columns = ["query_id","doc_id","relevance"]
corpus["doc_id"]    = corpus["doc_id"].astype(str)
queries["query_id"] = queries["query_id"].astype(str)
qrels["query_id"]   = qrels["query_id"].astype(str)
queries = queries[queries["query_id"].isin(set(qrels["query_id"]))].reset_index(drop=True)
print(f"\u2705 data ready: {len(corpus)} documents, {len(queries)} queries\n")

MODELS = {"distilbert-base-uncased":"distilbert-base-uncased",
          "BAAI/bge-large-en-v1.5":"BAAI__bge-large-en-v1.5"}
timings = {}
for n,(name,safe) in enumerate(MODELS.items(),1):
    print(f"--- Model {n} of 2: {name} ---")
    print("  \u23f3 downloading model (first time only) ...")
    model = SentenceTransformer(name)
    print("  \u2705 model ready. DOCUMENT vectors:")
    t0=time.time()
    doc_emb = model.encode(corpus["text"].tolist(), show_progress_bar=True)
    timings[name]=round(time.time()-t0,2)
    print("  \u2705 done. QUERY vectors:")
    qry_emb = model.encode(queries["text"].tolist(), show_progress_bar=True)
    np.save(f"out/embeddings_{safe}.npy", doc_emb)
    np.save(f"out/query_embeddings_{safe}.npy", qry_emb)
    print(f"  \u2705 saved {doc_emb.shape} docs, {qry_emb.shape} queries, {timings[name]}s\n")

# Save the id order so the notebook can align these vectors no matter what.
json.dump(corpus["doc_id"].tolist(),  open("out/doc_ids.json","w"))
json.dump(queries["query_id"].tolist(), open("out/query_ids.json","w"))
json.dump(timings, open("out/task2_timings.json","w"), indent=2)
print("\u2705 STEP 2 DONE - all embeddings + id lists saved.")


## STEP 3 of 3 — Download the results as a zip


In [ ]:
print("STEP 3 of 3 - packaging files ...")
import shutil
shutil.make_archive("task2_embeddings","zip","out")
from google.colab import files
print("\u2705 STEP 3 DONE - your browser will download task2_embeddings.zip now.")
print("\U0001f449 Upload that zip file to Claude and it will push it to your repo.")
files.download("task2_embeddings.zip")
